# City Search Agent — Ground Truth Evaluations

This notebook demonstrates evaluation of an agentic application with ground truth using Amazon Bedrock AgentCore Evaluations:

| Interface | When to use |
|---|---|
| **EvaluationClient** | You already have agent sessions in CloudWatch. Evaluate specific sessions against reference inputs. |
| **OnDemandEvaluationDatasetRunner** | You have a test dataset. You want to invoke the agent for every scenario and evaluate the results. |

### The City Search Agent

We'll deploy the same **City Search Agent** used throughout this workshop — a Strands agent that helps users find:
- Population data for US cities
- Land area (in square miles) for US cities

The agent uses a `web_search` tool (DuckDuckGo) to look up current information, then returns structured answers with `<pop>` and `<area>` XML tags for programmatic parsing.

### What You'll Learn
- How to use `EvaluationClient` to evaluate existing agent sessions logged in Amazon CloudWatch with ground-truth references
- How to use `OnDemandEvaluationDatasetRunner` to run automated dataset evaluations
- How to create **custom LLM-as-a-Judge evaluators** with ground truth placeholders
- How to interpret evaluation results across built-in evaluators (Correctness, GoalSuccessRate, Trajectory)

### Tutorial Details

| Information | Details |
|---|---|
| Agent framework | Strands Agents |
| Runtime | Amazon Bedrock AgentCore Runtime |
| Evaluation SDK | `bedrock-agentcore` |
| AWS services | AgentCore Runtime, AgentCore Evaluations, CloudWatch Logs |

### Prerequisites
- Python 3.10+
- AWS credentials with permissions for AgentCore, Lambda, CloudWatch, ECR, IAM
- Docker running locally (for agent container image build)

## Step 1: Install Dependencies

In [1]:
!pip install -r requirements.txt -q

## Step 2: Configuration

Import libraries and configure your AWS session.

In [4]:
import boto3
import json
import time
import uuid
from datetime import timedelta
from boto3.session import Session
from IPython.display import display, Markdown

boto_session = Session()
REGION = boto_session.region_name

print(f"Region  : {REGION}")

Region  : us-west-2


## Step 3: Deploy the City Search Agent

We deploy the same City Search Agent used in module 03 (Agentic Metrics) and module 05-02 (AgentCore) to **AgentCore Runtime**.

The agent uses:
- **web_search tool**: Queries DuckDuckGo for current city population and area data
- **Amazon Nova Micro**: Optimized for low latency and cost
- **XML output format**: `<pop>` and `<area>` tags for programmatic parsing

The deployment steps are:
1. **Write agent file** — create `citysearch.py` with the BedrockAgentCoreApp entrypoint
2. **Configure** — set up ECR, IAM roles, and agent configuration
3. **Launch** — build container via CodeBuild and deploy to AgentCore Runtime

In [5]:
%%writefile citysearch.py
from botocore.config import Config
from ddgs import DDGS
from strands import Agent, tool
from strands.models import BedrockModel

from bedrock_agentcore.runtime import BedrockAgentCoreApp
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

# Custom config for Bedrock — short timeouts, no retries
quick_config = Config(
    connect_timeout=5,
    read_timeout=20,
    retries={"max_attempts": 0}
)


@tool
def web_search(topic: str) -> str:
    """Search DuckDuckGo for a given topic.
    Return a string listing the top 5 results including the url, title, and description of each result.
    """
    try:
        results = DDGS(timeout=5).text(topic, region=region, max_results=5)
        return results if results else "No results found."
    except Exception as e:
        return f"Search error: {str(e)}"


SYSTEM_PROMPT = """You are a helpful city information assistant.

You help users find population and land area data for US cities.
Always use the web_search tool to look up current data — do not guess.
Be concise, professional, and friendly.

After your response, also include your answer in 'pop' and 'area' XML tags
for programmatic processing.
The values in the XML tags should only be numbers, no words or commas."""

chatbot_model = BedrockModel(
    model_id="us.amazon.nova-2-lite-v1:0",
    boto_client_config=quick_config
)
chatbot = Agent(tools=[web_search], model=chatbot_model, system_prompt=SYSTEM_PROMPT)

# Initialize the AgentCore Runtime App
app = BedrockAgentCoreApp()


@app.entrypoint
def invoke(payload):
    """AgentCore Runtime entrypoint function"""
    user_input = payload.get("prompt", "")
    response = chatbot(user_input)
    return response.message["content"][0]["text"]


if __name__ == "__main__":
    app.run()

Overwriting citysearch.py


In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime = Runtime()
agentcore_runtime.configure(
    entrypoint="citysearch.py",
    agent_name="citysearch_groundtruth_eval",
    region=REGION,
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    non_interactive=True,
)
print("Configuration complete.")

print("\nDeploying City Search Agent ...")
print("  This takes ~5 minutes on first run (image build + push + runtime creation).")
print()

_launch = agentcore_runtime.launch(auto_update_on_conflict=True)

print(f"\nLaunch complete.")
print(f"  agent_id  : {_launch.agent_id}")
print(f"  agent_arn : {_launch.agent_arn}")

In [ ]:
import time

print("Waiting for agent to reach READY status ...")

_POLL_INTERVAL = 15   # seconds between status checks
_MAX_WAIT      = 600  # 10-minute timeout

_elapsed = 0
while _elapsed < _MAX_WAIT:
    _status_result = agentcore_runtime.status()
    _agent_info    = _status_result.agent or {}
    _agent_status  = _agent_info.get("status", "UNKNOWN")
    print(f"  [{_elapsed:>3}s] status = {_agent_status}")

    if _agent_status in ("READY", "ACTIVE"):
        print(f"\nAgent is {_agent_status}. Proceeding.")
        break
    if _agent_status in ("FAILED", "CREATE_FAILED", "UPDATE_FAILED"):
        raise RuntimeError(
            f"Agent deployment failed with status '{_agent_status}'.\n"
            f"Details: {_agent_info}"
        )

    time.sleep(_POLL_INTERVAL)
    _elapsed += _POLL_INTERVAL
else:
    raise TimeoutError(
        f"Agent did not reach READY status within {_MAX_WAIT}s. "
        "Check the AgentCore console for details."
    )

In [7]:
AGENT_ID     = _launch.agent_id
AGENT_ARN    = _launch.agent_arn
CW_LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{AGENT_ID}-DEFAULT"

In [ ]:
# Persist agent info
%store AGENT_ID
%store AGENT_ARN
%store CW_LOG_GROUP
%store REGION

In [2]:
# # Uncomment this if you need to restore the saved variables 
# %store -r AGENT_ID
# %store -r AGENT_ARN
# %store -r CW_LOG_GROUP
# %store -r REGION

In [ ]:
print(f"AGENT_ID     : {AGENT_ID}")
print(f"AGENT_ARN    : {AGENT_ARN}")
print(f"CW_LOG_GROUP : {CW_LOG_GROUP}")

In [12]:
from botocore.config import Config
agentcore_config = Config(read_timeout=180, retries={"max_attempts": 2})
agentcore_client = boto3.client(
    "bedrock-agentcore", region_name=REGION, config=agentcore_config
)

## Step 4: Invoke the Agent to Generate Sessions

Before we can evaluate, we need agent sessions with CloudWatch spans. We'll invoke the agent
for several city search scenarios and record the session IDs for use with `EvaluationClient`.

Each session corresponds to one evaluation scenario.

In [13]:
def invoke_agent(prompt: str, session_id: str) -> str:
    """Send a single prompt to the city search agent and return its text response."""
    resp = agentcore_client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_ARN,
        qualifier="DEFAULT",
        runtimeSessionId=session_id,
        payload=json.dumps({"prompt": prompt}),
    )
    response_body = resp["response"].read()
    response_data = json.loads(response_body)
    return response_data if isinstance(response_data, str) else str(response_data)


def run_session(turns: list[str], session_prefix: str) -> str:
    """Invoke a multi-turn session and return its session ID."""
    session_id = f"{session_prefix}-{uuid.uuid4()}"
    print(f"Session: {session_id}")
    for turn_input in turns:
        print(f"  > {turn_input[:70]}")
        response = invoke_agent(turn_input, session_id)
        print(f"  < {response[:100]}")
    return session_id

In [14]:
# --- Single-turn sessions ---

print("=== Single-Turn Sessions ===")

session_new_york = run_session(
    ["How many people live in New York, and what's the area of the city in square miles?"],
    "city-new-york"
)

session_phoenix = run_session(
    ["What is the population and land area of Phoenix, AZ?"],
    "city-phoenix"
)

session_chicago = run_session(
    ["How many people live in Chicago, IL, and what's the area of the city in square miles?"],
    "city-chicago"
)

print("\nSingle-turn sessions created.")

=== Single-Turn Sessions ===
Session: city-new-york-f32ebecd-0f3f-462d-80e2-2636a983a19f
  > How many people live in New York, and what's the area of the city in s
  < Based on the information I found, here are the details for New York City:

**Population**: As of 202
Session: city-phoenix-f9a3e6c5-29f6-4421-a40e-2a4ff1baedac
  > What is the population and land area of Phoenix, AZ?
  < Based on the latest data:

**Population**: Phoenix, AZ has an estimated population of approximately 
Session: city-chicago-f4f5e3b1-66d6-4086-8e33-d55ccd24ba3a
  > How many people live in Chicago, IL, and what's the area of the city i
  < Based on the information I've gathered, here's the data for Chicago, IL:

The population of Chicago 

Single-turn sessions created.


In [15]:
# --- Multi-turn session: Comparing cities ---

print("=== Multi-Turn Session: City Comparison ===")

session_city_comparison = run_session(
    [
        "What is the population and area of Los Angeles, CA?",
        "Now look up the same info for Houston, TX.",
        "Which of the two cities has a higher population density?",
    ],
    "city-comparison-session"
)

print("\nMulti-turn session created.")

=== Multi-Turn Session: City Comparison ===
Session: city-comparison-session-8127ad8d-37b3-439a-8930-d83ca22eeefa
  > What is the population and area of Los Angeles, CA?
  < Based on the search results, here is the information for Los Angeles, CA:

**Population:** Approxima
  > Now look up the same info for Houston, TX.
  < Based on the search results, here is the information for Houston, TX:

**Population:** Approximately
  > Which of the two cities has a higher population density?
  < To determine which city has a higher population density, we need to calculate the population density

Multi-turn session created.


In [16]:
# --- Multi-turn session: Regional exploration ---

print("=== Multi-Turn Session: Regional Exploration ===")

session_regional = run_session(
    [
        "What is the population and area of Seattle, WA?",
        "How about San Francisco, CA?",
        "And Denver, CO?",
    ],
    "regional-exploration"
)

print("\nAll sessions created. Waiting 60s for CloudWatch log ingestion...")
time.sleep(60)
print("Ready to evaluate.")

=== Multi-Turn Session: Regional Exploration ===
Session: regional-exploration-ed52dda6-c11b-4a23-8ed6-7d57ab107dfe
  > What is the population and area of Seattle, WA?
  < Based on the information retrieved, here are the key details for Seattle, WA:

**Population**: Seatt
  > How about San Francisco, CA?
  < Based on the retrieved data, here are the key details for San Francisco, CA:

**Population**: The ci
  > And Denver, CO?
  < Based on the retrieved data, here are the key details for Denver, CO:

**Population**: Denver has a 

All sessions created. Waiting 60s for CloudWatch log ingestion...
Ready to evaluate.


## Step 5: EvaluationClient — Evaluate Existing Sessions

`EvaluationClient` is the right tool when you **already have agent sessions** logged in CloudWatch and you want to test them against your ground truth in an ad-hoc manner.
It looks up the agent's spans for a given `session_id` and runs evaluators against them. For these evaluations, you can pass in an expected response, assertions, and expected trajectory. You can use the built-in evaluators as well as custom evaluators.

### Ground-Truth Reference Inputs

`ReferenceInputs` lets you supply optional ground truth:

| Field | Evaluators that use it | Description |
|---|---|---|
| `expected_response` | `Builtin.Correctness` | The ideal response text |
| `expected_trajectory` | `Builtin.TrajectoryExactOrderMatch`, `Builtin.TrajectoryInOrderMatch`, `Builtin.TrajectoryAnyOrderMatch` | Ordered list of tool names |
| `assertions` | `Builtin.GoalSuccessRate` | Free-text assertions the session should satisfy |

Evaluators that don't need ground truth (`Helpfulness`, `ResponseRelevance`) can be included in the same call.
Each evaluator only reads the fields it needs.

### Create Custom (LLM-as-a-Judge) Evaluators

In addition to built-in evaluators, you can define your own evaluation criteria using
**LLM-as-a-Judge custom evaluators**. These accept natural language instructions that
can reference **ground truth placeholders** automatically substituted at evaluation time.

#### Ground truth placeholders

| Level | Available placeholders |
|---|---|
| **TRACE** | `{context}`, `{assistant_turn}`, `{expected_response}` |
| **SESSION** | `{context}`, `{available_tools}`, `{actual_tool_trajectory}`, `{expected_tool_trajectory}`, `{assertions}` |

For example, a trace-level evaluator comparing response similarity would include
`{assistant_turn}` and `{expected_response}` in its instructions. When the evaluator runs,
the service substitutes those placeholders with the actual agent output and the
`expectedResponse` from `ReferenceInputs`.

#### What we'll create

| Evaluator | Level | Placeholders | Description |
|---|---|---|---|
| `CityResponseSimilarity` | TRACE | `{assistant_turn}`, `{expected_response}` | How closely the agent's city data matches the expected answer |
| `CityAssertionChecker` | SESSION | `{actual_tool_trajectory}`, `{expected_tool_trajectory}`, `{assertions}` | Whether the agent called the right tools and satisfied all session assertions |

In [17]:
import uuid

_SUFFIX = uuid.uuid4().hex[:8]
_cp = boto3.client("bedrock-agentcore-control", region_name=REGION)

# ---------------------------------------------------------------------------
# Trace-level: CityResponseSimilarity
# Compares the agent's response to the expected_response reference input.
# {assistant_turn} -> actual agent output
# {expected_response} -> expectedResponse field in ReferenceInputs
# ---------------------------------------------------------------------------
print("Creating CityResponseSimilarity (TRACE) ...")
_resp_sim = _cp.create_evaluator(
    evaluatorName=f"CityResponseSimilarity_{_SUFFIX}",
    level="TRACE",
    evaluatorConfig={
        "llmAsAJudge": {
            "instructions": (
                "Compare the agent's response with the expected response about city data.\n"
                "Agent response: {assistant_turn}\n"
                "Expected response: {expected_response}\n\n"
                "Rate how closely the agent's response matches the expected response. "
                "Focus on whether the population numbers and area figures are accurate."
            ),
            "ratingScale": {
                "numerical": [
                    {
                        "value": 0.0,
                        "label": "not_similar",
                        "definition": "Response has significantly different population or area numbers from the expected response.",
                    },
                    {
                        "value": 0.5,
                        "label": "partially_similar",
                        "definition": "Response has one correct figure but the other is wrong or missing.",
                    },
                    {
                        "value": 1.0,
                        "label": "highly_similar",
                        "definition": "Response is semantically equivalent — population and area figures are close to expected values.",
                    },
                ]
            },
            "modelConfig": {
                "bedrockEvaluatorModelConfig": {
                    "modelId": "us.amazon.nova-2-lite-v1:0",
                    "inferenceConfig": {"maxTokens": 512},
                }
            },
        }
    },
)
CUSTOM_RESPONSE_SIMILARITY_ID = _resp_sim["evaluatorId"]
print(f"  evaluatorId : {CUSTOM_RESPONSE_SIMILARITY_ID}")

# ---------------------------------------------------------------------------
# Session-level: CityAssertionChecker
# Evaluates tool trajectory compliance and assertion satisfaction.
# {actual_tool_trajectory}   -> tools the agent actually called
# {expected_tool_trajectory} -> expectedTrajectory from ReferenceInputs
# {assertions}               -> assertions list from ReferenceInputs
# ---------------------------------------------------------------------------
print("\nCreating CityAssertionChecker (SESSION) ...")
_assert_chk = _cp.create_evaluator(
    evaluatorName=f"CityAssertionChecker_{_SUFFIX}",
    level="SESSION",
    evaluatorConfig={
        "llmAsAJudge": {
            "instructions": (
                "Evaluate whether the agent fulfilled the session requirements.\n\n"
                "Expected tool trajectory: {expected_tool_trajectory}\n"
                "Actual tool trajectory: {actual_tool_trajectory}\n"
                "Assertions to verify: {assertions}\n\n"
                "Score the agent on how well it followed the expected tool trajectory "
                "and satisfied every listed assertion."
            ),
            "ratingScale": {
                "numerical": [
                    {
                        "value": 0.0,
                        "label": "failed",
                        "definition": "Agent did not follow the trajectory and failed most assertions.",
                    },
                    {
                        "value": 0.5,
                        "label": "partial",
                        "definition": "Agent partially followed the trajectory or satisfied only some assertions.",
                    },
                    {
                        "value": 1.0,
                        "label": "passed",
                        "definition": "Agent followed the expected trajectory and satisfied all assertions.",
                    },
                ]
            },
            "modelConfig": {
                "bedrockEvaluatorModelConfig": {
                    "modelId": "us.amazon.nova-2-lite-v1:0",
                    "inferenceConfig": {"maxTokens": 512},
                }
            },
        }
    },
)
CUSTOM_ASSERTION_CHECKER_ID = _assert_chk["evaluatorId"]
print(f"  evaluatorId : {CUSTOM_ASSERTION_CHECKER_ID}")

print(f"\nCustom evaluators ready:")
print(f"  CityResponseSimilarity (TRACE)   : {CUSTOM_RESPONSE_SIMILARITY_ID}")
print(f"  CityAssertionChecker   (SESSION) : {CUSTOM_ASSERTION_CHECKER_ID}")

Creating CityResponseSimilarity (TRACE) ...
  evaluatorId : CityResponseSimilarity_97db3e15-qX3R8q44BV

Creating CityAssertionChecker (SESSION) ...
  evaluatorId : CityAssertionChecker_97db3e15-aILiOJ37l4

Custom evaluators ready:
  CityResponseSimilarity (TRACE)   : CityResponseSimilarity_97db3e15-qX3R8q44BV
  CityAssertionChecker   (SESSION) : CityAssertionChecker_97db3e15-aILiOJ37l4


In [18]:
from bedrock_agentcore.evaluation import EvaluationClient, ReferenceInputs

eval_client = EvaluationClient(region_name=REGION)

print(f"EvaluationClient initialised (region={REGION})")
print(f"  {CUSTOM_RESPONSE_SIMILARITY_ID} -> TRACE  (custom: CityResponseSimilarity)")
print(f"  {CUSTOM_ASSERTION_CHECKER_ID} -> SESSION (custom: CityAssertionChecker)")

EvaluationClient initialised (region=us-west-2)
  CityResponseSimilarity_97db3e15-qX3R8q44BV -> TRACE  (custom: CityResponseSimilarity)
  CityAssertionChecker_97db3e15-aILiOJ37l4 -> SESSION (custom: CityAssertionChecker)


In [19]:
# Helper function for printing
def display_eval_results(label: str, results: list) -> None:
    """Pretty-print EvaluationClient results as a markdown table."""
    rows = ["| Evaluator | Value | Label | Explanation |",
            "|---|---|---|---|"]
    for r in results:
        evaluator = r.get("evaluatorId", "")[:40]
        value = str(r.get("value", r.get("score", "N/A")))
        lbl = str(r.get("label", r.get("rating", "")))
        explanation = (r.get("explanation", r.get("reason", "")) or "")[:120].replace("\n", " ")
        error_code = r.get("errorCode")
        if error_code:
            lbl = f"ERR:{error_code}"
            explanation = (r.get("errorMessage", "") or "")[:120]
        rows.append(f"| `{evaluator}` | {value} | {lbl} | {explanation} |")

    if len(rows) == 2:  # only header rows, no data
        rows.append("| No results — session may be too recent or spans not yet visible | | | |")

    md = f"### {label}\n\n" + "\n".join(rows)
    display(Markdown(md))

### 5a. Single-Turn: New York — Correctness + Helpfulness + Custom ResponseSimilarity

We evaluate the New York city search response against known ground truth data from our `city_pop.csv` dataset using `Builtin.Correctness` and the custom `CityResponseSimilarity` evaluator. Both measure factual accuracy but use different scoring rubrics.

In [20]:
new_york_results = eval_client.run(
    evaluator_ids=[
        "Builtin.Correctness",           # TRACE: compares with provided expected response
        "Builtin.Helpfulness",           # TRACE: no ground truth needed
        "Builtin.ResponseRelevance",     # TRACE: no ground truth needed
        CUSTOM_RESPONSE_SIMILARITY_ID,   # TRACE: custom — uses {assistant_turn} + {expected_response}
    ],
    session_id=session_new_york,
    agent_id=AGENT_ID,
    look_back_time=timedelta(hours=2),
    reference_inputs=ReferenceInputs(
        expected_response="New York has a population of approximately 8,478,072 people and a land area of about 300.5 square miles.",
    ),
)

display_eval_results("New York — Correctness + Quality + Custom ResponseSimilarity", new_york_results)

### New York — Correctness + Quality + Custom ResponseSimilarity

| Evaluator | Value | Label | Explanation |
|---|---|---|---|
| `Builtin.Correctness` | 1.0 | Correct | The agent response provides population and area figures for New York City that are close to the expected response but wi |
| `Builtin.Helpfulness` | 0.83 | Very Helpful | The user's goal is straightforward: obtain the population and area (in square miles) of New York City. The assistant's r |
| `Builtin.ResponseRelevance` | 1.0 | Completely Yes | The user asked two specific questions: (1) How many people live in New York, and (2) what's the area of the city in squa |
| `CityResponseSimilarity_97db3e15-qX3R8q44` | 0.5 | partially_similar | The agent's response provides a population figure of approximately 8.8 million, which is slightly higher than the expect |

### 5b. Single-Turn: Phoenix — Assertions + Trajectory + Custom AssertionChecker

This cell runs both built-in trajectory evaluators **and** the custom `CityAssertionChecker`
(which uses `{actual_tool_trajectory}`, `{expected_tool_trajectory}`, and `{assertions}` placeholders)
plus the custom `CityResponseSimilarity` for the response. This lets you compare built-in vs. custom
scoring side by side.

In [21]:
phoenix_results = eval_client.run(
    evaluator_ids=[
        "Builtin.GoalSuccessRate",           # SESSION: built-in assertion evaluator
        "Builtin.TrajectoryExactOrderMatch", # SESSION: built-in trajectory evaluator
        "Builtin.TrajectoryAnyOrderMatch",   # SESSION: built-in trajectory evaluator
        "Builtin.Correctness",               # TRACE: built-in response accuracy
        CUSTOM_RESPONSE_SIMILARITY_ID,       # TRACE (custom): {assistant_turn} + {expected_response}
    ],
    session_id=session_phoenix,
    agent_id=AGENT_ID,
    look_back_time=timedelta(hours=2),
    reference_inputs=ReferenceInputs(
        expected_trajectory=["web_search"],
        assertions=[
            "Agent called web_search to look up Phoenix population and area data",
            "Agent reported a population close to 1,673,164 for Phoenix, AZ",
            "Agent reported a land area close to 518 square miles for Phoenix, AZ",
        ],
        expected_response="Phoenix, AZ has a population of approximately 1,673,164 and a land area of about 518 square miles.",
    ),
)

display_eval_results("Phoenix — Built-in + Custom ResponseSimilarity", phoenix_results)

### Phoenix — Built-in + Custom ResponseSimilarity

| Evaluator | Value | Label | Explanation |
|---|---|---|---|
| `Builtin.GoalSuccessRate` | 1.0 | Yes | The agent successfully completed all three assertions: (1) It called web_search multiple times to look up Phoenix popula |
| `Builtin.TrajectoryExactOrderMatch` | 0.0 | No | Length mismatch: Expected 1 tools ['web_search'], but got 3 tools ['web_search', 'web_search', 'web_search'] |
| `Builtin.TrajectoryAnyOrderMatch` | 1.0 | Yes | Any-order match: All expected tools ['web_search'] found in actual ['web_search', 'web_search', 'web_search'] |
| `Builtin.Correctness` | 1.0 | Correct | The agent response provides population and land area figures for Phoenix, AZ that are very close to the expected respons |
| `CityResponseSimilarity_97db3e15-qX3R8q44` | 1.0 | highly_similar | The agent's response has a population figure that is slightly higher than the expected response (1,706,000 vs. 1,673,164 |

### 5c. Single-Turn: Chicago — Factual Correctness

Factual data retrieval scenarios are well-suited for `Builtin.Correctness` combined with
`Builtin.GoalSuccessRate`. The expected_response provides the ground truth figures.

In [22]:
chicago_results = eval_client.run(
    evaluator_ids=[
        "Builtin.Correctness",
        "Builtin.GoalSuccessRate",
    ],
    session_id=session_chicago,
    agent_id=AGENT_ID,
    look_back_time=timedelta(hours=2),
    reference_inputs=ReferenceInputs(
        expected_response="Chicago, IL has a population of approximately 2,721,308 and a land area of about 227.7 square miles.",
        assertions=[
            "Agent called web_search for Chicago population and area",
            "Agent reported a population close to 2,721,308",
            "Agent reported a land area close to 227.7 square miles",
        ],
    ),
)

display_eval_results("Chicago — Correctness + GoalSuccessRate", chicago_results)

### Chicago — Correctness + GoalSuccessRate

| Evaluator | Value | Label | Explanation |
|---|---|---|---|
| `Builtin.Correctness` | 1.0 | Correct | The agent response provides population and area data for Chicago, IL. Comparing to the expected response:  Population: A |
| `Builtin.GoalSuccessRate` | 1.0 | Yes | The agent successfully completed all three assertions: (1) It called web_search multiple times to gather information abo |

### 5d. Multi-Turn: City Comparison Session (3 turns) + Custom AssertionChecker

For multi-turn sessions, `EvaluationClient` fetches all spans for the session and evaluates
the complete conversation. The trajectory and assertions apply across all turns.

This scenario exercises the custom `CityAssertionChecker` evaluator (SESSION level),
which uses `{actual_tool_trajectory}`, `{expected_tool_trajectory}`, and `{assertions}`
placeholders. A 3-turn session with distinct queries per turn gives the evaluator
a rich trajectory to compare against the expected sequence.

In [23]:
comparison_results = eval_client.run(
    evaluator_ids=[
        "Builtin.GoalSuccessRate",
        "Builtin.TrajectoryExactOrderMatch",
        "Builtin.TrajectoryInOrderMatch",
        "Builtin.TrajectoryAnyOrderMatch",
        "Builtin.Helpfulness",
        CUSTOM_ASSERTION_CHECKER_ID,   # SESSION (custom): trajectory + assertions
    ],
    session_id=session_city_comparison,
    agent_id=AGENT_ID,
    look_back_time=timedelta(hours=2),
    reference_inputs=ReferenceInputs(
        expected_trajectory=["web_search", "web_search"],
        assertions=[
            "Agent looked up population and area for Los Angeles in turn 1",
            "Agent looked up population and area for Houston in turn 2",
            "Agent compared population density between the two cities in turn 3",
            "Agent correctly identified that Los Angeles has a higher population density than Houston",
        ],
    ),
)

display_eval_results("City Comparison — Multi-Turn (3 turns) + Custom AssertionChecker", comparison_results)

### City Comparison — Multi-Turn (3 turns) + Custom AssertionChecker

| Evaluator | Value | Label | Explanation |
|---|---|---|---|
| `Builtin.GoalSuccessRate` | 1.0 | Yes | The agent successfully completed all required tasks across the three turns:  1. Turn 1: Agent performed web searches for |
| `Builtin.TrajectoryExactOrderMatch` | 0.0 | No | Length mismatch: Expected 2 tools ['web_search', 'web_search'], but got 6 tools ['web_search', 'web_search', 'web_search |
| `Builtin.TrajectoryInOrderMatch` | 1.0 | Yes | In-order match: All expected tools ['web_search', 'web_search'] found in order within actual ['web_search', 'web_search' |
| `Builtin.TrajectoryAnyOrderMatch` | 1.0 | Yes | Any-order match: All expected tools ['web_search', 'web_search'] found in actual ['web_search', 'web_search', 'web_searc |
| `Builtin.Helpfulness` | 0.83 | Very Helpful | The user's goal is straightforward: obtain the population and land area of Los Angeles, CA. The assistant's response dir |
| `Builtin.Helpfulness` | 0.83 | Very Helpful | The user's goal is clear: obtain the population and land area for Houston, TX, following their previous request for the  |
| `Builtin.Helpfulness` | 0.83 | Very Helpful | The user's goal is to determine which of the two cities (Los Angeles and Houston) has a higher population density. The a |
| `CityAssertionChecker_97db3e15-aILiOJ37l4` | 0.5 |  | The agent followed an unexpected trajectory by performing six web searches instead of the expected two. Despite this, th |

## Step 6: OnDemandEvaluationDatasetRunner — Automated Dataset Evaluation

`OnDemandEvaluationDatasetRunner` is the right tool when you have a **test dataset** and want to:
1. Automatically invoke your agent for each scenario
2. Collect CloudWatch spans
3. Run evaluators against each scenario's results

This is ideal for regression testing, CI/CD pipelines, and batch evaluation against curated datasets.

### Dataset structure

A dataset consists of **scenarios**, each with one or more **turns**. Optional ground-truth fields:
- `Turn.expected_response` — per-turn expected answer
- `PreDefinedScenario.expected_trajectory` — ordered list of tool names
- `PreDefinedScenario.assertions` — session-level assertions

### How OnDemandEvaluationDatasetRunner works

```
For each scenario:
  1. Create a new session ID
  2. Call your agent_invoker function for each turn
  3. Wait for CloudWatch spans to appear (evaluation_delay_seconds)
  4. Submit spans + ground truth to the evaluation service
  5. Collect and return results
```

In [24]:
from bedrock_agentcore.evaluation import (
    AgentInvokerInput,
    AgentInvokerOutput,
    CloudWatchAgentSpanCollector,
    Dataset,
    EvaluationRunConfig,
    OnDemandEvaluationDatasetRunner,
    EvaluatorConfig,
    Turn,
    PredefinedScenario,
)

In [25]:
def agent_invoker(invoker_input: AgentInvokerInput) -> AgentInvokerOutput:
    """
    Called by OnDemandEvaluationDatasetRunner once per turn. Invoke the city search
    agent and return the text response.
    """
    payload = invoker_input.payload
    body = {"prompt": payload} if isinstance(payload, str) else payload

    resp = agentcore_client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_ARN,
        qualifier="DEFAULT",
        runtimeSessionId=invoker_input.session_id,
        payload=json.dumps(body),
    )

    response_body = resp["response"].read()
    response_data = json.loads(response_body)
    return AgentInvokerOutput(
        agent_output=response_data if isinstance(response_data, str) else str(response_data)
    )

### 6a. Define the Evaluation Dataset

We define scenarios inline using ground truth data from the `city_pop.csv` dataset. A mix of single-turn and multi-turn scenarios exercises different aspects of the agent.

In [26]:
dataset = Dataset(
    scenarios=[
        # --- Single-turn: New York population and area ---
        PredefinedScenario(
            scenario_id="city-new-york",
            turns=[
                Turn(
                    input="How many people live in New York, and what's the area of the city in square miles?",
                    expected_response="New York has a population of approximately 8,478,072 and a land area of about 300.5 square miles.",
                )
            ],
            expected_trajectory=["web_search"],
            assertions=[
                "Agent called web_search to look up New York population and area",
                "Agent reported a population close to 8,478,072",
                "Agent reported a land area close to 300.5 square miles",
            ],
        ),

        # --- Single-turn: Los Angeles ---
        PredefinedScenario(
            scenario_id="city-los-angeles",
            turns=[
                Turn(
                    input="What is the population and land area of Los Angeles, CA?",
                    expected_response="Los Angeles has a population of approximately 3,878,704 and a land area of about 469.5 square miles.",
                )
            ],
            expected_trajectory=["web_search"],
            assertions=[
                "Agent called web_search to look up Los Angeles data",
                "Agent reported a population close to 3,878,704",
                "Agent reported a land area close to 469.5 square miles",
            ],
        ),

        # --- Single-turn: Houston ---
        PredefinedScenario(
            scenario_id="city-houston",
            turns=[
                Turn(
                    input="How many people live in Houston, TX, and what's the area of the city in square miles?",
                    expected_response="Houston has a population of approximately 2,390,125 and a land area of about 640.4 square miles.",
                )
            ],
            expected_trajectory=["web_search"],
            assertions=[
                "Agent called web_search to look up Houston data",
                "Agent reported a population close to 2,390,125",
                "Agent reported a land area close to 640.4 square miles",
            ],
        ),

        # --- Single-turn: San Francisco ---
        PredefinedScenario(
            scenario_id="city-san-francisco",
            turns=[
                Turn(
                    input="What is the population and land area of San Francisco, CA?",
                    expected_response="San Francisco has a population of approximately 827,526 and a land area of about 46.9 square miles.",
                )
            ],
            expected_trajectory=["web_search"],
            assertions=[
                "Agent called web_search to look up San Francisco data",
                "Agent reported a population close to 827,526",
                "Agent reported a land area close to 46.9 square miles",
            ],
        ),

        # --- Multi-turn: Comparing two cities ---
        PredefinedScenario(
            scenario_id="city-comparison-multi",
            turns=[
                Turn(
                    input="What is the population and area of Seattle, WA?",
                    expected_response="Seattle has a population of approximately 780,995 and a land area of about 83.8 square miles.",
                ),
                Turn(
                    input="Now look up the same for Denver, CO.",
                    expected_response="Denver has a population of approximately 729,019 and a land area of about 153.1 square miles.",
                ),
                Turn(
                    input="Which city has a higher population density?",
                    expected_response="Seattle has a higher population density than Denver because it has a larger population in a much smaller area.",
                ),
            ],
            expected_trajectory=["web_search", "web_search"],
            assertions=[
                "Agent called web_search twice across the conversation for Seattle and Denver",
                "Agent correctly reported population and area for both cities",
                "Agent correctly identified Seattle as having higher population density",
            ],
        ),
    ]
)

print(f"Dataset contains {len(dataset.scenarios)} scenarios.")

Dataset contains 5 scenarios.


### 6b. Configure and Run OnDemandEvaluationDatasetRunner

In [27]:
# Span collector: polls CloudWatch for OTel spans emitted by the agent
span_collector = CloudWatchAgentSpanCollector(
    log_group_name=CW_LOG_GROUP,
    region=REGION,
    max_wait_seconds=180,
    poll_interval_seconds=15,
)

# Evaluator level cache — built-ins + custom evaluators
EVALUATOR_LEVELS = {
    "Builtin.GoalSuccessRate":            "SESSION",
    "Builtin.TrajectoryExactOrderMatch":  "SESSION",
    "Builtin.TrajectoryInOrderMatch":     "SESSION",
    "Builtin.TrajectoryAnyOrderMatch":    "SESSION",
    "Builtin.Correctness":                "TRACE",
}
# Custom evaluators
EVALUATOR_LEVELS[CUSTOM_RESPONSE_SIMILARITY_ID] = "TRACE"
EVALUATOR_LEVELS[CUSTOM_ASSERTION_CHECKER_ID]   = "SESSION"

# Evaluator configuration — mix of built-in and custom evaluators
config = EvaluationRunConfig(
    evaluator_config=EvaluatorConfig(
        evaluator_ids=[
            "Builtin.Correctness",               # TRACE — expected_response
            "Builtin.GoalSuccessRate",            # SESSION — assertions
            "Builtin.TrajectoryExactOrderMatch",  # SESSION — expected_trajectory
            "Builtin.TrajectoryInOrderMatch",     # SESSION — expected_trajectory
            "Builtin.TrajectoryAnyOrderMatch",    # SESSION — expected_trajectory
            CUSTOM_RESPONSE_SIMILARITY_ID,       # TRACE (custom) — {assistant_turn} + {expected_response}
            CUSTOM_ASSERTION_CHECKER_ID,         # SESSION (custom) — {actual_tool_trajectory} + {assertions}
        ]
    ),
    evaluation_delay_seconds=180,
    max_concurrent_scenarios=3,
)

runner = OnDemandEvaluationDatasetRunner(region=REGION)
runner._evaluator_level_cache.update(EVALUATOR_LEVELS)

print("OnDemandEvaluationDatasetRunner configured. Starting evaluation...")
print(f"  Scenarios : {len(dataset.scenarios)}")
print(f"  Evaluators: {len(config.evaluator_config.evaluator_ids)} "
      f"(5 built-in + 2 custom)")
print(f"  Delay     : {config.evaluation_delay_seconds}s (waiting for CloudWatch ingestion)")

OnDemandEvaluationDatasetRunner configured. Starting evaluation...
  Scenarios : 5
  Evaluators: 7 (5 built-in + 2 custom)
  Delay     : 180s (waiting for CloudWatch ingestion)


In [28]:
# Run the evaluation.
# OnDemandEvaluationDatasetRunner will:
#   1. Invoke agent_invoker for each turn in each scenario
#   2. Wait evaluation_delay_seconds for CloudWatch ingestion
#   3. Submit spans to the evaluation service
#   4. Return aggregated results

eval_result = runner.run(
    config=config,
    dataset=dataset,
    agent_invoker=agent_invoker,
    span_collector=span_collector,
)

completed = sum(1 for sr in eval_result.scenario_results if sr.status == "COMPLETED")
failed = sum(1 for sr in eval_result.scenario_results if sr.status == "FAILED")
print(f"\nEvaluation complete: {completed} completed, {failed} failed out of {len(eval_result.scenario_results)} scenarios.")


Evaluation complete: 5 completed, 0 failed out of 5 scenarios.


### 6c. Inspect Results

In [29]:
def display_runner_results(eval_result) -> None:
    """Display OnDemandEvaluationDatasetRunner results as a markdown table per scenario."""
    for sr in eval_result.scenario_results:
        if sr.status == "FAILED":
            display(Markdown(f"**Scenario `{sr.scenario_id}`** — FAILED: {sr.error}"))
            continue

        rows = ["| Evaluator | Value | Label | Explanation |",
                "|---|---|---|---|"]
        for er in sr.evaluator_results:
            for res in er.results:
                value = str(res.get("value", res.get("score", "N/A")))
                lbl = str(res.get("label", res.get("rating", "")))
                explanation = (res.get("explanation", "") or "")[:130].replace("\n", " ")
                error_code = res.get("errorCode")
                if error_code:
                    lbl = f"ERR:{error_code}"
                    explanation = (res.get("errorMessage", "") or "")[:130]
                rows.append(f"| `{er.evaluator_id[:40]}` | {value} | {lbl} | {explanation} |")

        md = f"### Scenario: `{sr.scenario_id}`\n\n" + "\n".join(rows)
        display(Markdown(md))


display_runner_results(eval_result)

### Scenario: `city-new-york`

| Evaluator | Value | Label | Explanation |
|---|---|---|---|
| `Builtin.Correctness` | 1.0 | Correct | The agent response provides population and area figures for New York City that are close to the expected response but with some di |
| `Builtin.GoalSuccessRate` | 1.0 | Yes | The agent successfully called web_search with the topic 'New York population and land area', satisfying the first assertion. The a |
| `Builtin.TrajectoryExactOrderMatch` | 1.0 | Yes | Exact match: Actual trajectory ['web_search'] matches expected trajectory ['web_search'] |
| `Builtin.TrajectoryInOrderMatch` | 1.0 | Yes | In-order match: All expected tools ['web_search'] found in order within actual ['web_search'] |
| `Builtin.TrajectoryAnyOrderMatch` | 1.0 | Yes | Any-order match: All expected tools ['web_search'] found in actual ['web_search'] |
| `CityResponseSimilarity_97db3e15-qX3R8q44` | 1.0 | highly_similar | The agent's response mentions a population of approximately 8.8 million and a land area of 302.6 square miles, which is slightly h |
| `CityAssertionChecker_97db3e15-aILiOJ37l4` | 1.0 | passed | The agent followed the expected tool trajectory by using the 'web_search' tool. It also satisfied all assertions by reporting a po |

### Scenario: `city-los-angeles`

| Evaluator | Value | Label | Explanation |
|---|---|---|---|
| `Builtin.Correctness` | 0.0 | Incorrect | The agent response provides significantly different factual information compared to the expected response. For population, the age |
| `Builtin.GoalSuccessRate` | 0.0 | No | The agent successfully called web_search to look up Los Angeles data. The agent reported a population of 3,902,440, which is close |
| `Builtin.TrajectoryExactOrderMatch` | 1.0 | Yes | Exact match: Actual trajectory ['web_search'] matches expected trajectory ['web_search'] |
| `Builtin.TrajectoryInOrderMatch` | 1.0 | Yes | In-order match: All expected tools ['web_search'] found in order within actual ['web_search'] |
| `Builtin.TrajectoryAnyOrderMatch` | 1.0 | Yes | Any-order match: All expected tools ['web_search'] found in actual ['web_search'] |
| `CityResponseSimilarity_97db3e15-qX3R8q44` | 0.5 | partially_similar | The agent's response provides a population of approximately 3,902,440 and a land area of 795.581 square miles. The expected respon |
| `CityAssertionChecker_97db3e15-aILiOJ37l4` | N/A | ERR:ValueError | No score found in evaluation result |

### Scenario: `city-houston`

| Evaluator | Value | Label | Explanation |
|---|---|---|---|
| `Builtin.Correctness` | 0.0 | Incorrect | The agent response provides significantly different factual information compared to the expected response. For population, the age |
| `Builtin.GoalSuccessRate` | 0.0 | No | The agent successfully called web_search to look up Houston data, satisfying the first assertion. However, the reported population |
| `Builtin.TrajectoryExactOrderMatch` | 1.0 | Yes | Exact match: Actual trajectory ['web_search'] matches expected trajectory ['web_search'] |
| `Builtin.TrajectoryInOrderMatch` | 1.0 | Yes | In-order match: All expected tools ['web_search'] found in order within actual ['web_search'] |
| `Builtin.TrajectoryAnyOrderMatch` | 1.0 | Yes | Any-order match: All expected tools ['web_search'] found in actual ['web_search'] |
| `CityResponseSimilarity_97db3e15-qX3R8q44` | 0.0 | not_similar | The agent's response has a population of 2,304,580, which is significantly different from the expected population of approximately |
| `CityAssertionChecker_97db3e15-aILiOJ37l4` | N/A | ERR:ValueError | No score found in evaluation result |

### Scenario: `city-san-francisco`

| Evaluator | Value | Label | Explanation |
|---|---|---|---|
| `Builtin.Correctness` | 1.0 | Correct | The agent response provides population and land area figures for San Francisco that are close to but slightly different from the e |
| `Builtin.GoalSuccessRate` | 1.0 | Yes | The agent successfully called web_search multiple times to look up San Francisco data, satisfying the first assertion. For the pop |
| `Builtin.TrajectoryExactOrderMatch` | 0.0 | No | Length mismatch: Expected 1 tools ['web_search'], but got 3 tools ['web_search', 'web_search', 'web_search'] |
| `Builtin.TrajectoryInOrderMatch` | 1.0 | Yes | In-order match: All expected tools ['web_search'] found in order within actual ['web_search', 'web_search', 'web_search'] |
| `Builtin.TrajectoryAnyOrderMatch` | 1.0 | Yes | Any-order match: All expected tools ['web_search'] found in actual ['web_search', 'web_search', 'web_search'] |
| `CityResponseSimilarity_97db3e15-qX3R8q44` | 0.0 | not_similar | The agent's response provides a population figure of 808,437 and a land area of 46.7 square miles, which are both lower than the e |
| `CityAssertionChecker_97db3e15-aILiOJ37l4` | 0.5 | partial | The agent executed the 'web_search' tool three times, which exceeds the expected trajectory of one 'web_search'. However, it is un |

### Scenario: `city-comparison-multi`

| Evaluator | Value | Label | Explanation |
|---|---|---|---|
| `Builtin.Correctness` | 1.0 | Correct | The agent response provides the core factual information requested: Seattle's population of 780,995 and land area of 83.84 square  |
| `Builtin.Correctness` | 0.0 | Incorrect | The agent response provides the population figure of 729,019 for Denver, CO, which matches the expected response exactly. However, |
| `Builtin.Correctness` | 1.0 | Correct | The agent response correctly identifies that Seattle has a higher population density than Denver, which matches the expected respo |
| `Builtin.GoalSuccessRate` | 1.0 | Yes | The agent successfully completed all three success assertions:  1. **Web search calls**: The agent called web_search multiple time |
| `Builtin.TrajectoryExactOrderMatch` | 0.0 | No | Length mismatch: Expected 2 tools ['web_search', 'web_search'], but got 6 tools ['web_search', 'web_search', 'web_search', 'web_se |
| `Builtin.TrajectoryInOrderMatch` | 1.0 | Yes | In-order match: All expected tools ['web_search', 'web_search'] found in order within actual ['web_search', 'web_search', 'web_sea |
| `Builtin.TrajectoryAnyOrderMatch` | 1.0 | Yes | Any-order match: All expected tools ['web_search', 'web_search'] found in actual ['web_search', 'web_search', 'web_search', 'web_s |
| `CityResponseSimilarity_97db3e15-qX3R8q44` | 1.0 | highly_similar | The agent's response matches the expected response closely. Both the population and land area figures are accurate and consistent  |
| `CityResponseSimilarity_97db3e15-qX3R8q44` | 0.5 | partially_similar | The agent's response provides a population number of approximately 729,019, which closely matches the expected response. However,  |
| `CityResponseSimilarity_97db3e15-qX3R8q44` | 1.0 | highly_similar | The agent's response includes the exact population and area figures for both Seattle and Denver, and provides the correct calculat |
| `CityAssertionChecker_97db3e15-aILiOJ37l4` | 0.5 | partial | The agent followed an unexpected tool trajectory by using 'web_search' six times instead of the expected two. Although the agent m |

In [30]:
# Aggregate summary: average score per evaluator across all scenarios
from collections import defaultdict

scores_by_evaluator = defaultdict(list)
for sr in eval_result.scenario_results:
    if sr.status != "COMPLETED":
        continue
    for er in sr.evaluator_results:
        for res in er.results:
            if "value" in res and res["value"] is not None and not res.get("errorCode"):
                scores_by_evaluator[er.evaluator_id].append(float(res["value"]))

print("\nEvaluator Summary (average score across all scenarios)")
print("=" * 60)
for evaluator_id, scores in sorted(scores_by_evaluator.items()):
    avg = sum(scores) / len(scores)
    print(f"  {evaluator_id:<45} avg={avg:.2f}  (n={len(scores)})")


Evaluator Summary (average score across all scenarios)
  Builtin.Correctness                           avg=0.57  (n=7)
  Builtin.GoalSuccessRate                       avg=0.60  (n=5)
  Builtin.TrajectoryAnyOrderMatch               avg=1.00  (n=5)
  Builtin.TrajectoryExactOrderMatch             avg=0.60  (n=5)
  Builtin.TrajectoryInOrderMatch                avg=1.00  (n=5)
  CityAssertionChecker_97db3e15-aILiOJ37l4      avg=0.67  (n=3)
  CityResponseSimilarity_97db3e15-qX3R8q44BV    avg=0.57  (n=7)


### 6d. Save Results to File

In [31]:
import os
from datetime import datetime

os.makedirs("results", exist_ok=True)
timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
results_path = f"results/groundtruth_eval_{timestamp}.json"

with open(results_path, "w") as f:
    json.dump(eval_result.model_dump(), f, indent=2, default=str)

print(f"Results saved to: {results_path}")

Results saved to: results/groundtruth_eval_20260415_195210.json


/var/folders/7h/g0rdp4_96lvg7gd2h2b8c0d00000gq/T/ipykernel_32539/3226257259.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")


## Step 7: Cleanup

Delete the agent runtime endpoint when you're done to avoid ongoing costs.

In [ ]:
# Uncomment to delete the agent runtime
# agentcore_runtime.delete()
# print("Agent runtime deleted.")

print("Cleanup skipped. Uncomment the cell above to delete the agent runtime.")

## Key Takeaways

| | EvaluationClient | OnDemandEvaluationDatasetRunner |
|---|---|---|
| **When to use** | You have existing sessions | You have a test dataset |
| **Best for** | Post-hoc analysis, debugging | Regression testing, CI/CD |
| **Input** | session_id | Dataset of scenarios |

### Built-in evaluator reference

| Evaluator | Level | Ground truth required |
|---|---|---|
| `Builtin.Correctness` | TRACE | `expected_response` |
| `Builtin.GoalSuccessRate` | SESSION | `assertions` |
| `Builtin.TrajectoryExactOrderMatch` | SESSION | `expected_trajectory` |
| `Builtin.TrajectoryInOrderMatch` | SESSION | `expected_trajectory` |
| `Builtin.TrajectoryAnyOrderMatch` | SESSION | `expected_trajectory` |


### Custom evaluator ground truth placeholders

Custom (LLM-as-a-judge) evaluators reference ground truth via placeholders in their `instructions`.

| Level | Placeholder | Filled from |
|---|---|---|
| TRACE | `{assistant_turn}` | Agent's actual response |
| TRACE | `{expected_response}` | `ReferenceInputs.expected_response` |
| TRACE | `{context}` | Session context |
| SESSION | `{actual_tool_trajectory}` | Tools called by the agent |
| SESSION | `{expected_tool_trajectory}` | `ReferenceInputs.expected_trajectory` |
| SESSION | `{assertions}` | `ReferenceInputs.assertions` |
| SESSION | `{available_tools}` | Tools available to the agent |